In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

spark.sql("USE loan_risk")
print("Config loaded")

In [0]:
df_spark = spark.table("loan_risk.silver_loans")
print(f"Rows   : {df_spark.count():,}")
print(f"Columns: {len(df_spark.columns)}")
display(df_spark.limit(3))

In [0]:
print("Converting to Pandas...")
pdf = df_spark.toPandas()
print(f"Rows: {len(pdf):,}")



In [0]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in ["home_ownership", "purpose", "addr_state", "emp_length"]:
    pdf[col + "_enc"] = le.fit_transform(pdf[col].astype(str))

print("Categorical columns encoded")
print(f"   home_ownership sample: {pdf['home_ownership'].unique()[:3]}")
print(f"   encoded sample       : {pdf['home_ownership_enc'].unique()[:3]}")

In [0]:
from sklearn.model_selection import train_test_split

# Columns we feed into the model
FEATURES = [
    "loan_amnt", "int_rate", "installment", "grade_num",
    "log_annual_inc", "dti", "dti_ratio", "credit_utilization",
    "log_loan_amnt", "term_months", "risk_score",
    "home_ownership_enc", "purpose_enc", "total_acc"
]

X = pdf[FEATURES]  # inputs  — what the model learns from
y = pdf["default"] # output  — what the model predicts

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f" Training rows : {X_train.shape[0]:,}")
print(f" Test rows     : {X_test.shape[0]:,}")
print(f" Features      : {len(FEATURES)}")
print(f" Default rate in test: {y_test.mean():.1%}")

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# Start MLflow experiment
mlflow.set_experiment("/loan_risk_experiment")

with mlflow.start_run(run_name="Logistic_Regression"):
    
    # Train model
    print("Training Logistic Regression...")
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    
    # Predict
    lr_proba = lr.predict_proba(X_test)[:, 1]
    lr_auc   = roc_auc_score(y_test, lr_proba)
    
    # Log to MLflow
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("auc", lr_auc)
    mlflow.sklearn.log_model(lr, "logistic_regression_model")
    
    print(f" Logistic Regression AUC: {lr_auc:.4f}")
    print(classification_report(y_test, lr.predict(X_test)))

In [0]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="Random_Forest"):
    
    # Train model
    print("Training Random Forest...")
    print("(This will take 3-5 minutes — 100 trees on 1M rows)")
    
    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1        # use all CPU cores
    )
    rf.fit(X_train, y_train)
    
    # Predict
    rf_proba = rf.predict_proba(X_test)[:, 1]
    rf_auc   = roc_auc_score(y_test, rf_proba)
    
    # Log to MLflow
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("auc", rf_auc)
    mlflow.sklearn.log_model(rf, "random_forest_model")
    
    print(f" Random Forest AUC: {rf_auc:.4f}")
    print(classification_report(y_test, rf.predict(X_test)))

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

with mlflow.start_run(run_name="LR_Improved"):

    # Pipeline: scale features first, then train
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight="balanced"  # pay attention to minority class
        ))
    ])

    print("Training improved Logistic Regression...")
    pipeline.fit(X_train, y_train)

    lr2_proba = pipeline.predict_proba(X_test)[:, 1]
    lr2_auc   = roc_auc_score(y_test, lr2_proba)

    mlflow.log_param("model_type", "LR_balanced_scaled")
    mlflow.log_metric("auc", lr2_auc)
    mlflow.sklearn.log_model(pipeline, "lr_improved_model")

    print(f" Improved LR AUC : {lr2_auc:.4f}")
    print(f" Previous LR AUC : {lr_auc:.4f}")
    print(f" Improvement     : +{lr2_auc - lr_auc:.4f}")
    print(classification_report(y_test, pipeline.predict(X_test)))

In [0]:
# Score every loan with best model (improved LR)
best_model = pipeline

pdf["default_probability"] = best_model.predict_proba(pdf[FEATURES])[:, 1]

# Risk label buckets
pdf["risk_label"] = pd.cut(
    pdf["default_probability"],
    bins=[0, 0.2, 0.4, 0.6, 1.0],
    labels=["Low", "Medium", "High", "Very High"]
).astype(str)

print("Risk label distribution:")
print(pdf["risk_label"].value_counts())

# Convert back to Spark
df_gold = spark.createDataFrame(pdf)

# Save as Gold Delta table
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("loan_risk.gold_loans_scored")
)

print(f"\n Gold table saved!")
print(f" Rows    : {df_gold.count():,}")
print(f" Columns : {len(df_gold.columns)}")

# Quick verify with SQL
display(spark.sql("""
    SELECT
        risk_label,
        COUNT(*)                                    AS loan_count,
        ROUND(AVG(default_probability) * 100, 1)   AS avg_default_prob_pct,
        ROUND(AVG(int_rate), 1)                     AS avg_interest_rate,
        ROUND(AVG(dti), 1)                          AS avg_dti
    FROM loan_risk.gold_loans_scored
    GROUP BY risk_label
    ORDER BY avg_default_prob_pct DESC
"""))